# Randolph et al. 2021 — download and convert (Science, doi:10.1126/science.abg0928)

The Zenodo archive (10.5281/zenodo.4273999, 36.6 GB, open access, CC-BY) holds the inputs of the authors' pipeline. We need only two files from it:
- `inputs/1_calculate_pseudobulk/mergedAllCells_withCellTypeIdents_CLEAN.rds`: Seurat object with raw UMI counts per cell, cell-type identities and `sample_condition` (donor + condition)
- `inputs/2_calculate_residuals_and_DE_analyses/individual_meta_data_for_GE_with_scaledCovars_with_geneProps.txt`: per-sample metadata, including `capture` (processing batch), `infection_status`, `indiv_ID` and ancestry

The archive is **streamed** and only these two files are written to disk, so the full 36.6 GB is never stored. The Seurat object is then converted with R into a MatrixMarket matrix plus CSV tables that Python can read.

In [1]:
from datetime import datetime
from zoneinfo import ZoneInfo
now = datetime.now(ZoneInfo("America/New_York"))  
print("start:", now.strftime("%H:%M:%S"))
import os, subprocess
RAND_DIR = "data/randolph2021"
os.makedirs(RAND_DIR, exist_ok=True)
ZENODO_URL = "https://zenodo.org/records/4273999/files/inputs.tar.gz?download=1"
WANTED = ["inputs/1_calculate_pseudobulk/mergedAllCells_withCellTypeIdents_CLEAN.rds",
          "inputs/2_calculate_residuals_and_DE_analyses/individual_meta_data_for_GE_with_scaledCovars_with_geneProps.txt"]

start: 00:20:56


## 1. Download the archive to disk (resumable), then extract the two files
Streaming failed when the server dropped the connection after 10 GB, and a stream cannot be resumed. The archive is therefore saved to disk first (36.6 GB); `curl -C -` resumes an interrupted download, and the loop retries until the file is complete. For a long download, running the same command in a terminal with `nohup` (see the reply) is more robust than a notebook cell.

In [2]:
now = datetime.now(ZoneInfo("America/New_York"))  
print("start:", now.strftime("%H:%M:%S"))
import time
ARCHIVE = os.path.join(RAND_DIR, "inputs.tar.gz")
TOTAL = 36_606_000_000          # approximate size in bytes (36.6 GB)

def remote_size(url):
    head = subprocess.run(["curl", "-sIL", url], capture_output=True, text=True).stdout
    sizes = [int(l.split(":")[1]) for l in head.splitlines() if l.lower().startswith("content-length")]
    return sizes[-1] if sizes else None

remote = remote_size(ZENODO_URL) or TOTAL
for attempt in range(50):                                  # resume until complete
    now = datetime.now(ZoneInfo("America/New_York"))  
    print("start:", now.strftime("%H:%M:%S"))
    have = os.path.getsize(ARCHIVE) if os.path.exists(ARCHIVE) else 0
    if have >= remote: break
    print(f"attempt {attempt + 1}: {have / 1e9:.1f} / {remote / 1e9:.1f} GB")
    subprocess.run(["curl", "-L", "-C", "-", "--retry", "10", "--retry-delay", "20", "-sS", "-o", ARCHIVE, ZENODO_URL])
    time.sleep(5)
print(f"archive: {os.path.getsize(ARCHIVE) / 1e9:.2f} GB (expected {remote / 1e9:.2f} GB)")
now = datetime.now(ZoneInfo("America/New_York"))  
print("start:", now.strftime("%H:%M:%S"))

start: 00:21:41
start: 00:21:42
attempt 1: 0.0 / 36.6 GB
start: 03:15:47
archive: 36.65 GB (expected 36.65 GB)
start: 03:15:47


## 1b. Extract only the two needed files from the archive on disk

In [3]:
subprocess.run(["tar", "-xzf", ARCHIVE, "--occurrence=1", "-C", RAND_DIR] + WANTED)
for w in WANTED:
    p = os.path.join(RAND_DIR, w)
    print(w, f"{os.path.getsize(p) / 1e9:.2f} GB" if os.path.exists(p) else "MISSING")

inputs/1_calculate_pseudobulk/mergedAllCells_withCellTypeIdents_CLEAN.rds 17.80 GB
inputs/2_calculate_residuals_and_DE_analyses/individual_meta_data_for_GE_with_scaledCovars_with_geneProps.txt 0.00 GB


## 2. Convert the Seurat object with R
Writes `counts.mtx` (genes x cells), `genes.csv`, `cells.csv` (per-cell metadata + cell type). Requires R with the `Seurat` (or `SeuratObject`) and `Matrix` packages; set `RSCRIPT` to the Rscript of your R environment.

In [6]:
!free -g

              total        used        free      shared  buff/cache   available
Mem:            377          17         339           0          20         357
Swap:             3           0           3


In [5]:
!which Rscript

~/miniconda/envs/renv/bin/Rscript


In [7]:
RSCRIPT = "~/miniconda/envs/renv/bin/Rscript"          # e.g. "~/miniconda/envs/renv/bin/Rscript"
OUT = os.path.join(RAND_DIR, "converted")
os.makedirs(OUT, exist_ok=True)
r_code = f"""
suppressPackageStartupMessages({{library(Matrix); library(SeuratObject)}})
obj <- readRDS("{RAND_DIR}/{WANTED[0]}")
obj <- tryCatch(UpdateSeuratObject(obj), error = function(e) obj)
counts <- tryCatch(GetAssayData(obj, assay = "RNA", layer = "counts"),
                   error = function(e) GetAssayData(obj, assay = "RNA", slot = "counts"))
meta <- obj@meta.data
meta$celltype <- as.character(Idents(obj))
meta$barcode <- colnames(counts)
writeMM(counts, "{OUT}/counts.mtx")
write.csv(data.frame(gene = rownames(counts)), "{OUT}/genes.csv", row.names = FALSE)
write.csv(meta, "{OUT}/cells.csv", row.names = FALSE)
cat("cells:", ncol(counts), " genes:", nrow(counts), "\\n")
cat("metadata columns:", paste(colnames(meta), collapse = ", "), "\\n")
"""
with open(os.path.join(OUT, "convert.R"), "w") as f: f.write(r_code)
if os.path.exists(os.path.join(OUT, "counts.mtx")):
    print("Already converted")
else:
    subprocess.run([os.path.expanduser(RSCRIPT), os.path.join(OUT, "convert.R")], check=True)

Warning message:
package ‘Matrix’ was built under R version 4.3.3 
Validating object structure
Updating object slots
Loading required package: Seurat
Ensuring keys are in the proper structure
Updating matrix keys for DimReduc ‘pca’
Updating matrix keys for DimReduc ‘umap’
Ensuring keys are in the proper structure
Ensuring feature names don't have underscores or pipes
Updating slots in RNA
Updating slots in SCT
Updating slots in integrated
Updating slots in integrated_nn
Setting default assay of integrated_nn to integrated
Updating slots in integrated_snn
Setting default assay of integrated_snn to integrated
Updating slots in pca
Updating slots in umap
Setting umap DimReduc to global
No assay information could be found for FindIntegrationAnchors
No assay information could be found for PairwiseIntegrateReference
Setting assay used for ScaleData.integrated to integrated
Setting assay used for RunPCA.integrated to integrated
Setting assay used for RunUMAP.integrated.pca to integrated
Setti

NULL
cells: 236993  genes: 19248 
metadata columns: orig.ident, nCount_RNA, nFeature_RNA, batchID, percent.mt, SOC_status, SOC_indiv_ID, SOC_infection_status, SOC_genetic_ancestry, CEU, YRI, nCount_SCT, nFeature_SCT, integrated_snn_res.0.5, cluster_IDs, celltype, sample_condition, barcode 


## 3. Inspect

In [8]:
import pandas as pd
cells = pd.read_csv(os.path.join(RAND_DIR, "converted", "cells.csv"))
print(cells.shape); print(cells.head(3).T)
for col in cells.columns:
    if cells[col].nunique() < 40:
        print(f"\n{col}:", cells[col].value_counts().head(15).to_dict())
meta_ind = pd.read_csv(os.path.join(RAND_DIR, WANTED[1]), sep=",")
print("\nper-sample metadata:", meta_ind.shape, "\ncolumns:", meta_ind.columns.tolist())
print(meta_ind.head(3).T)

(236993, 18)
                                             0                       1  \
orig.ident                               B1_c1                   B1_c1   
nCount_RNA                                1695                    2116   
nFeature_RNA                               714                     822   
batchID                                  B1_c1                   B1_c1   
percent.mt                            1.828909                3.922495   
SOC_status                             singlet                 singlet   
SOC_indiv_ID                          HMN83551                HMN83551   
SOC_infection_status                        NI                      NI   
SOC_genetic_ancestry                       EUR                     EUR   
CEU                                    0.99999                 0.99999   
YRI                                    0.00001                 0.00001   
nCount_SCT                                1641                    1826   
nFeature_SCT             